# 10 - Sensitivity to the DFM design parameters (workflow step 11c)

One-at-a-time sweeps around the nominal design, each evaluated nominally **and** on the same
100 Monte Carlo realizations (the risk tolerance only matters under uncertainty):

* support horizon $H_f\in\{15,30,60,120\}$ min
* risk tolerance $\varepsilon\in\{0.01,0.05,0.10,0.20\}$
* initial health $SOH_0\in\{1.00,0.95,0.90,0.85\}$
* degradation allowance $N_{ref}\in\{0.25,0.5,1.0,1.5\}$ reference cycles per day

Outputs: `output/results/results_sensitivity_trials.csv`, `output/results/results_sensitivity_summary.csv`,
`output/experiment_figures/Fig_ablation_sensitivity.pdf`.

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dfm_acdc.config import RESULTS_DIR, FIG_DIR, MCParams, DFMParams, BESSParams
from dfm_acdc.mc import run_jobs
from dfm_acdc import plotstyle

plotstyle.apply_style()
PRE_PATH = os.path.join(RESULTS_DIR, "precompute_network_forecast.csv")
mc = MCParams()
if os.environ.get("DFM_SMOKE"):
    mc.n_trials = 2          # quick end-to-end smoke test only
d0, b0 = DFMParams(), BESSParams()
SWEEPS = {
    "H_f_h": ("dfm", [0.25, 0.5, 1.0, 2.0]),
    "eps_risk": ("dfm", [0.01, 0.05, 0.10, 0.20]),
    "soh_0": ("bess", [1.00, 0.95, 0.90, 0.85]),
    "N_ref_cycles_per_day": ("dfm", [0.25, 0.5, 1.0, 1.5]),
}
COL = {"H_f_h": "H_f_h", "eps_risk": "eps_risk", "soh_0": "soh_0", "N_ref_cycles_per_day": "N_ref"}  # summary columns
NOMINAL = {"H_f_h": d0.H_f_h, "eps_risk": d0.eps_risk, "soh_0": b0.soh_0, "N_ref_cycles_per_day": d0.N_ref_cycles_per_day}

In [ ]:
# RERUN = True recomputes all 1212 closed-loop days (about 1 h with 8 workers); False reloads the
# trials saved by the full run (results/results_sensitivity_trials.csv) to regenerate tables/figure.
RERUN = False
TRIALS_PATH = os.path.join(RESULTS_DIR, "results_sensitivity_trials.csv")
if RERUN or not os.path.exists(TRIALS_PATH):
    jobs = []
    for par, (grp, vals) in SWEEPS.items():
        for v in vals:
            if np.isclose(v, NOMINAL[par]):
                continue                                   # nominal design = DFM runs of notebooks 08/09
            for tr in [None] + list(range(mc.n_trials)):
                jobs.append({"strategy": "DFM", "trial": tr, grp: {par: v}, "tag": f"{par}={v}"})
    t0 = time.perf_counter()
    out = run_jobs(jobs, PRE_PATH, mc.seed, mc.n_trials)
    sens = pd.DataFrame([o["summary"] for o in out])
    print(f"{len(out)} runs in {time.perf_counter()-t0:.0f} s")
    # nominal design: MC trials from notebook 08 + nominal run from notebook 09
    main = pd.read_csv(os.path.join(RESULTS_DIR, "results_mc_trials.csv"))
    abl = pd.read_csv(os.path.join(RESULTS_DIR, "results_ablation_trials.csv"))
    base = pd.concat([main[main.strategy == "DFM"], abl[(abl.strategy == "DFM") & (abl.tag == "abl_nominal")]])
    base["tag"] = "nominal"
    sens = pd.concat([sens, base], ignore_index=True)
    sens.to_csv(TRIALS_PATH, index=False)
else:
    sens = pd.read_csv(TRIALS_PATH)
    print(f"loaded {len(sens)} saved runs (full run: 1212 new closed-loop days, 3709 s on 8 workers)")

In [ ]:
M = ["cost_corr_EUR", "E_curt_kWh", "dSOH_rainflow", "N_EFC", "E_dev_kWh", "n_viol_intervals", "soc_final",
     "mean_P_safe_dis_kW", "mean_P_safe_ch_kW", "mean_P_flex_dis_kW", "share_lim_dis_deg", "share_lim_dis_en",
     "share_lim_dis_net"]
rows = []
for par, (grp, vals) in SWEEPS.items():
    for v in vals:
        sel = sens[np.isclose(sens[COL[par]], v)]
        for k, p in NOMINAL.items():                   # other parameters at nominal
            if k != par:
                sel = sel[np.isclose(sel[COL[k]], p)]
        d = sel[sel.trial >= 0]; nom = sel[sel.trial < 0]
        r = {"parameter": par, "value": v, "n_trials": len(d)}
        for m in M:
            r[f"mc_mean_{m}"] = d[m].mean()
            r[f"nom_{m}"] = nom[m].iloc[0] if len(nom) else np.nan
        r["mc_P95_E_dev_kWh"] = np.percentile(d.E_dev_kWh, 95)
        r["mc_P_viol_trials"] = float((d.n_viol_intervals > 0).mean())
        rows.append(r)
summ = pd.DataFrame(rows)
summ.to_csv(os.path.join(RESULTS_DIR, "results_sensitivity_summary.csv"), index=False)
print(summ[["parameter", "value", "n_trials", "mc_mean_cost_corr_EUR", "mc_mean_E_curt_kWh", "mc_mean_dSOH_rainflow",
            "mc_mean_E_dev_kWh", "mc_P_viol_trials", "mc_mean_mean_P_safe_dis_kW", "mc_mean_share_lim_dis_deg",
            "nom_cost_corr_EUR"]].round(4).to_string(index=False))

## Figure 8 -- ablation and parameter sensitivity

In [ ]:
ablsum = pd.read_csv(os.path.join(RESULTS_DIR, "results_ablation_summary.csv"), index_col=0)
FIG_WIDTH, FIG_HEIGHT = plotstyle.IEEE_PAGE_WIDTH_IN, 4.6
fig = plt.figure(figsize=(FIG_WIDTH, FIG_HEIGHT))
gs = fig.add_gridspec(2, 3, hspace=0.55, wspace=0.42)
ax = fig.add_subplot(gs[0, 0])
mets = [("rel_mc_cost_corr_EUR_pct", "Cost"), ("rel_mc_E_curt_kWh_pct", r"$E_{curt}$"),
        ("rel_mc_dSOH_rainflow_pct", r"$\Delta SOH$"), ("rel_mc_E_dev_kWh_pct", "Imbal.")]
x = np.arange(len(mets))
for i, cfg in enumerate(("M1", "M2", "M3")):
    vals = [ablsum.loc[cfg, m] for m, _ in mets]
    ax.bar(x + (i - 1) * 0.26, vals, width=0.24, color=plotstyle.STRATEGY_COLORS[cfg], label=plotstyle.STRATEGY_LABELS[cfg])
ax.axhline(0, color=plotstyle.INK_MUTED, lw=0.6)
ax.set_xticks(x); ax.set_xticklabels([l for _, l in mets], fontsize=7)
ax.set_ylabel(r"Change vs $\mathcal{M}_0$ [%]"); ax.set_yscale("symlog", linthresh=10); ax.set_ylim(-15, 2000)
ax.legend(frameon=False, fontsize=6, loc="upper left")
titles = {"H_f_h": r"Support horizon $H_f$ [h]", "eps_risk": r"Risk tolerance $\varepsilon$",
          "soh_0": r"Initial health $SOH_0$", "N_ref_cycles_per_day": r"Allowance $N_{ref}$ [cycles/day]"}
series = [("mc_mean_cost_corr_EUR", "Cost", plotstyle.CAT["blue"]), ("mc_mean_dSOH_rainflow", r"$\Delta SOH$", plotstyle.CAT["orange"]),
          ("mc_mean_E_dev_kWh", "Uncovered imbalance", plotstyle.CAT["aqua"]),
          ("mc_mean_mean_P_safe_dis_kW", r"mean $P^{dis}_{safe}$", plotstyle.CAT["violet"])]
axes = [fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[0, 2]), fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])]
for ax, par in zip(axes, SWEEPS):
    d = summ[summ.parameter == par].sort_values("value")
    ref = d[np.isclose(d.value, NOMINAL[par])].iloc[0]
    for col, lab, c in series:
        ax.plot(d.value, 100 * (d[col] - ref[col]) / ref[col], "-o", ms=3.5, color=c, label=lab)
    ax.axhline(0, color=plotstyle.INK_MUTED, lw=0.6); ax.axvline(NOMINAL[par], color=plotstyle.INK_MUTED, lw=0.6, ls=":")
    ax.set_xlabel(titles[par], fontsize=7.5); ax.set_ylabel("Change vs nominal [%]", fontsize=7.5)
    if par == "H_f_h":
        ax.set_xscale("log", base=2); ax.set_xticks(d.value); ax.set_xticklabels([f"{v:g}" for v in d.value])
axl = fig.add_subplot(gs[1, 2]); axl.axis("off")
axl.legend(*axes[0].get_legend_handles_labels(), frameon=False, fontsize=7, loc="center")
for a, lab in zip([fig.axes[0]] + axes, "abcde"):
    a.text(-0.32, 1.08, f"({lab})", transform=a.transAxes, fontsize=8, va="top")
fig.savefig(os.path.join(FIG_DIR, "Fig_ablation_sensitivity.pdf"), format="pdf", bbox_inches="tight")
plt.show()